<a href="https://colab.research.google.com/github/Cory-Suhr/rush-sales-analysis-final-project/blob/cleaning-sales-dataset/Rush_Sales_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GB885 Final Project

## Rush Sales Analysis

Cory Suhr

## Business Problem: You work as a sales analyst for RUSH, a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. The company stores its raw sales data as a collection of three tables:

TABLE_PRODUCTS
TABLE_RETAILER
TABLE_SALES
The data includes the number of units sold, the total sales revenue, the location of the sales, the type of product sold, as well as other relevant information. (For data field definitions and explanations, see the data dictionary.) The data is "raw," meaning it has not been cleaned and probably contains errors that need to be addressed.

The VP of US Sales has tasked you with analyzing sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth. For example, you may want to look for trends or insights in seasonality, retailers, locations, or sales methods. Take initiative to apply your creativity and curiosity to this data.

In addition, she has asked you to answer the following business questions:

What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
What state had the highest sales (in dollars) of women's products in 2021? How much was it?
What state had the highest sales (in dollars) of men's products in 2021? How much was it?
What retailer purchased the most units in 2021? In 2020?

## Import Libraries

In [ ]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## Need to load Google Drive to get the files

In [ ]:
# Load google drive
from google.colab import drive
drive.mount('/content/drive')

## Load the Data

In [ ]:
# Loading the data
products = pd.read_csv('/content/drive/MyDrive/TABLE_PRODUCTS_885.csv', sep='|')
retailer = pd.read_csv('/content/drive/MyDrive/TABLE_RETAILER_885.csv')
sales = pd.read_csv('/content/drive/MyDrive/TABLE_SALES_885.csv')

## Initial Data Inspection

In [ ]:
# Looking at some basics for  products

products.head()

In [ ]:
## Products variable types
products.info()

In [ ]:
# Basics of retailor
retailer.head()

In [ ]:
# Retailer variable types.
retailer.info()

In [ ]:
# Sales basics
sales.head()

In [ ]:
# Sales variable types
sales.info()

## Merge the Data Sets

In [ ]:
# Merging products and sales
sales_products = pd.merge(sales, products, on='PRODUCT_ID', how='left')

In [ ]:
# Checking my work
sales_products.head()

In [ ]:
# Merging sales_products with retailer
combined_df = pd.merge(sales_products, retailer, on='RETAILER_ID', how='left')

In [ ]:
# Check final merger
combined_df.head()

## Looking at the combined dataset for some basics:
Month, Day and Year could be deleted once invoice date is converted to datetime


*   Units sold should be numeric
*   PRICE_PER unit 2 null values
* Retailer, Region, State and City each have 1 null (Drop this row.)
* Sales_Method has 'Oolet' that shoule be 'Outlet'
* PRICE_PER_UNIT has 99999
*



In [ ]:
# Looking at the variable types
combined_df.info()

In [ ]:
# Further looks
combined_df.describe()

In [ ]:
# Looking for null values
combined_df.isnull().sum()

In [ ]:
# Looking at the rows with null values for PRICE_PER_UNIT
combined_df[combined_df['PRICE_PER_UNIT'].isnull()]


In [ ]:
# Looking at null values for RETAILER
combined_df[combined_df['RETAILER'].isnull()]

In [ ]:
# Verifying there this is the only entry with the RETAILER_ID with 999999999
combined_df[combined_df['RETAILER_ID'] == '999999999']

# With the null values in Retailer, region, state and city. And no way to determine the retailer based on the retailer ID going to drop this row from the dataframe

In [ ]:
# Dropping row with index 1534
combind_df = combined_df.drop(index=1534)

In [ ]:
# Looking for Non_traditional categorical data
cat_var = list(combined_df.select_dtypes(include=['object']).columns)
# View unique values for each categorical variable
for column in cat_var:
    print(column)
    print(combined_df[column].unique())


# Look for Duplicate Values

In [ ]:
# Checking for duplicates
combined_df.duplicated().sum()

# Check for Erroneous Data

In [ ]:
# Check for vaule count of erroneous data
# List of categorical variables
cat_var = list(combined_df.select_dtypes(include=['object']).columns)

# Loop through each categorical variable and print the value counts
for column in cat_var:
    print(column)
    print(combined_df[[column]].value_counts())


# Look for Outliers Using the Inner Quantile Ranges (IQR)

In [ ]:
# Write a function to calculate the IQR and print rows with values that fall outside that IQR

def count_iqr_outliers(df, column):
  #define q1
  q1 = df[column].quantile(0.25)
  #define q3
  q3 = df[column].quantile(0.75)
  # define iqr
  iqr = q3- q1
  # define the threshold
  l_threshold = q1 - 1.5 * iqr
  u_threshold = q3 + 1.5 * iqr
  # define outliers
  outliers = (df[column] < l_threshold) | (df[column] > u_threshold)
  # Count the number of True values (outliers)
  return outliers.sum()

In [ ]:
# iterate over the numerical columns of the dataframe
num_var = list(combined_df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
    print(f'{column} : {count_iqr_outliers(combined_df, column)}')

In [ ]:
# Looking into outliers
count_iqr_outliers(combined_df, 'PRICE_PER_UNIT')

## Data Cleaning
-Convert UNITS_SOLD to numeris
- Replace 2 null values in PRICE_PER_UNIT
-Replace the 99999 in PRICE_PER_UNIT
- Replace 'Oolet' with 'Outlet'

In [ ]:
# Converting Units Sold to numeric
combined_df['UNITS_SOLD'] = pd.to_numeric(combined_df['UNITS_SOLD'], errors='coerce')

In [ ]:
# Convert Invoice date to datetime
combined_df['INVOICE_DATE'] = pd.to_datetime(combined_df['INVOICE_DATE'])

In [ ]:
# Checking my work
combined_df.info()

In [ ]:
# Drop Month, Day and Year columns as they provide duplicated information contained in the invoice date
combined_df = combined_df.drop(columns=['MONTH', 'DAY', 'YEAR'])

In [ ]:
# Checking myy work
combined_df.head()

## All null values and the 99999 value in price per unit are with product 20. find the median and mean price for the product and replace the null and erroneous values

In [ ]:
# Finding mean and median for product ID 20
mean_price = combined_df[combined_df['PRODUCT_ID'] == 20]['PRICE_PER_UNIT'].mean()
median_price = combined_df[combined_df['PRODUCT_ID'] == 20]['PRICE_PER_UNIT'].median()

print(mean_price)
print(median_price)

## It seems the 99999 is skewing the mean. I will use the median to repalce the null values and the 99999

In [ ]:
# Replacing null values in price per unit with 44.0

combined_df['PRICE_PER_UNIT'] = combined_df['PRICE_PER_UNIT'].fillna(44.0)

In [ ]:
 # Replacing 99999 with the median of 44.0
combined_df['PRICE_PER_UNIT'] = combined_df['PRICE_PER_UNIT'].replace(99999.000000, 44.0)

## Replacing the 'Ootlet' with 'Oulet'

In [ ]:
# Replace sales method Ootlet with Oulet
combined_df['SALES_METHOD'] = combined_df['SALES_METHOD'].replace('Ootlet', 'Outlet')

In [ ]:
# Checking my work
combined_df.describe()

In [ ]:
# Checking all variable for sales method
combined_df['SALES_METHOD'].unique()

In [ ]:
# Double cheking for null values
combined_df.isnull().sum()

In [ ]:
# Looking at the null values
combined_df[combined_df['RETAILER'].isnull()]

In [ ]:
# Dropping index 1534 again
combined_df = combined_df.drop(index=1534)

In [ ]:
# Checking for null values
combined_df.isnull().sum()

## Exploratory Data Analysis

## VP Business Questions